# ThermoTech — Notebook 04: Feature Engineering & ML Handoff

**Goal:** Convert the outputs of Notebooks 01–03 into a clean, numeric, ML-ready dataset for the ML teammate.

This notebook **does not train XGBoost**. It prepares the data for training and creates a labeling template.

### Important truth about labels

The current GeoFlare repository explicitly requires team-assigned labels:

- `0` = `PERSISTENT_SOURCE`
- `1` = `OTHER_ANOMALY`
- `2` = `NEW_ABNORMAL_EVENT`

These labels are **not available from FIRMS/OSM automatically**. This notebook therefore does **not invent labels** or use the current rule-based classifier as ground truth.

Instead, it creates:

1. `geoflare_ml_ready_unlabeled.csv` — clean features + metadata, no target
2. `labeled_hotspots_template.csv` — backend-compatible training template with a blank `label` column
3. `geoflare_feature_dictionary.csv` — feature definitions
4. `geoflare_ml_ready_numeric.csv` — numeric ML table with a blank target

### Handoff philosophy

> **Feature engineering here, supervised learning in the partner's notebook/script.**

The dataset keeps useful identifiers such as coordinates and acquisition time for manual labeling, while the final model feature columns are numeric.

## 0. Pipeline position

```text
NASA FIRMS
    ↓
Notebook 01 — ingest / clean / explore
    ↓
Notebook 02 — historical baseline / persistence / anomaly
    ↓
Notebook 03 — OSM geographic context
    ↓
THIS NOTEBOOK — feature engineering + ML handoff
    ↓
ML teammate — manual labels → XGBoost → evaluation → SHAP
    ↓
trained model → backend classifier → dashboard
```

Notebook 03 already produced the enriched recent detections and OSM context files. The existing backend training script expects a labeled CSV containing the core features:

`frp, confidence, historical_count, historical_frp_mean, historical_frp_max, persistence_days, industrial_distance, residential_distance, road_distance, label`.

In [1]:
# Install once if needed:
# %pip install pandas numpy matplotlib

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Imports successful.")

Imports successful.


In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RECENT_OSM_FILE = PROCESSED_DIR / "firms_recent_scored_osm.csv"
BASELINE_OSM_FILE = PROCESSED_DIR / "firms_baseline_osm.csv"
CONTEXT_FILE = PROCESSED_DIR / "osm_context_features.csv"

if not RECENT_OSM_FILE.exists():
    raise FileNotFoundError(
        f"Missing Notebook 03 output: {RECENT_OSM_FILE}"
    )

if not BASELINE_OSM_FILE.exists():
    raise FileNotFoundError(
        f"Missing Notebook 03 output: {BASELINE_OSM_FILE}"
    )

recent_osm = pd.read_csv(RECENT_OSM_FILE)
baseline_osm = pd.read_csv(BASELINE_OSM_FILE)

context = (
    pd.read_csv(CONTEXT_FILE)
    if CONTEXT_FILE.exists()
    else pd.DataFrame()
)

print(f"Recent OSM-enriched rows: {len(recent_osm):,}")
print(f"Baseline OSM-enriched rows: {len(baseline_osm):,}")
print(f"OSM context rows: {len(context):,}")

Recent OSM-enriched rows: 1,112
Baseline OSM-enriched rows: 4,855
OSM context rows: 30


## 2. Validate the input schema

We expect the recent dataset to contain the FIRMS thermal fields plus Notebook 02 historical features and Notebook 03 OSM features.

If a required field is absent, the notebook stops instead of silently creating a wrong feature.

In [3]:
required_recent = {
    "latitude", "longitude", "acq_date", "acq_time",
    "frp", "confidence",
    "lat_grid", "lon_grid",
    "detection_count", "frp_mean",
    "active_days", "persistence_ratio",
    "has_baseline", "frp_zscore",
    "near_industrial", "nearest_industrial_m",
    "near_residential", "nearest_residential_m",
    "near_agriculture", "near_forest", "near_commercial",
    "osm_query_status",
}

missing_recent = sorted(required_recent - set(recent_osm.columns))

if missing_recent:
    raise ValueError(
        "Notebook 03 output is missing required columns: "
        + ", ".join(missing_recent)
    )

print("Input schema validated.")

Input schema validated.


## 3. Create stable observation IDs and clean basic fields

Each row represents a recent FIRMS thermal detection.

We keep the original coordinates and timestamp as metadata so the ML teammate can inspect and label examples manually.

In [4]:
df = recent_osm.copy()

df["source_row_id"] = np.arange(len(df))

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["frp"] = pd.to_numeric(df["frp"], errors="coerce")

df["acq_datetime"] = pd.to_datetime(
    df["acq_date"].astype(str) + " " +
    df["acq_time"].astype(str).str.zfill(4),
    format="%Y-%m-%d %H%M",
    errors="coerce",
)

df = df.dropna(
    subset=["source_row_id", "latitude", "longitude", "frp", "acq_datetime"]
).copy()

print(f"Rows after basic cleaning: {len(df):,}")
print(f"Duplicate source_row_id values: {df['source_row_id'].duplicated().sum()}")
print(f"Missing timestamps: {df['acq_datetime'].isna().sum()}")

Rows after basic cleaning: 1,112
Duplicate source_row_id values: 0
Missing timestamps: 0


## 4. Encode FIRMS confidence safely

FIRMS `confidence` is categorical (`n`, `l`, `h`), not a continuous measurement.

The existing backend training script expects a numeric `confidence` feature, so we create a documented ordinal encoding:

- `n` → `0`
- `l` → `1`
- `h` → `2`

The original value is preserved separately as `confidence_raw`.

This is an engineering compatibility encoding, not a claim that the confidence categories are a physically linear scale.

In [5]:
df["confidence_raw"] = (
    df["confidence"]
    .astype(str)
    .str.strip()
    .str.lower()
)

confidence_map = {
    "n": 0,
    "l": 1,
    "h": 2,
}

df["confidence"] = df["confidence_raw"].map(confidence_map)

unknown_confidence = sorted(
    df.loc[df["confidence"].isna(), "confidence_raw"].dropna().unique()
)

if unknown_confidence:
    raise ValueError(
        f"Unexpected FIRMS confidence values: {unknown_confidence}"
    )

print("Confidence encoding:")
display(
    df[["confidence_raw", "confidence"]]
    .drop_duplicates()
    .sort_values("confidence")
)

Confidence encoding:


,confidence_raw,confidence
0,n,0
11,l,1
288,h,2


## 5. Build the historical features

These come directly from Notebook 02's grid-cell baseline:

- historical detection count
- historical mean FRP
- historical maximum FRP
- active historical days
- persistence ratio
- whether a historical baseline exists
- FRP anomaly z-score

For a **new location with no historical baseline**, historical counts/statistics are set to `0`. That is meaningful here because Notebook 02 explicitly identifies those cells as having no prior baseline.

The anomaly z-score remains missing when a usable historical standard deviation was unavailable.

In [6]:
df["historical_count"] = pd.to_numeric(
    df["detection_count"], errors="coerce"
).fillna(0)

df["historical_frp_mean"] = pd.to_numeric(
    df["frp_mean"], errors="coerce"
).fillna(0)

historical_lookup = (
    baseline_osm[
        ["lat_grid", "lon_grid", "frp_max"]
    ]
    .drop_duplicates(["lat_grid", "lon_grid"])
    .rename(columns={"frp_max": "historical_frp_max"})
)

df = df.merge(
    historical_lookup,
    on=["lat_grid", "lon_grid"],
    how="left",
    validate="many_to_one",
)

df["historical_frp_max"] = pd.to_numeric(
    df["historical_frp_max"], errors="coerce"
).fillna(0)

df["persistence_days"] = pd.to_numeric(
    df["active_days"], errors="coerce"
).fillna(0)

df["persistence_ratio"] = pd.to_numeric(
    df["persistence_ratio"], errors="coerce"
).fillna(0)

df["frp_zscore"] = pd.to_numeric(
    df["frp_zscore"], errors="coerce"
)

df["has_baseline"] = (
    df["has_baseline"].fillna(False).astype(bool)
)

print("Historical features created.")

Historical features created.


## 6. Build OSM features

Notebook 03 queried OSM for a deliberately small representative set of cells.

**Important:** `not_queried` is not the same as “no industrial/residential feature exists”.

Therefore we keep:

- OSM presence flags as numeric 0/1
- nearest distances as missing (`NaN`) when no valid distance is available
- `osm_queried` as a separate indicator

XGBoost can handle missing numeric values; the ML teammate can also choose an explicit imputation strategy.

In [7]:
osm_boolean_cols = [
    "near_industrial",
    "near_agriculture",
    "near_forest",
    "near_residential",
    "near_commercial",
]

for col in osm_boolean_cols:
    df[col] = df[col].fillna(False).astype(int)

osm_distance_map = {
    "industrial_distance": "nearest_industrial_m",
    "residential_distance": "nearest_residential_m",
}

for new_col, old_col in osm_distance_map.items():
    df[new_col] = pd.to_numeric(df[old_col], errors="coerce")

df["osm_queried"] = (
    df["osm_query_status"].fillna("not_queried") != "not_queried"
).astype(int)

# Road distance was not collected by Notebook 03.
# Do NOT fabricate it from another OSM category.
df["road_distance"] = np.nan

print("OSM features created.")
print(
    "Rows with successful/usable OSM query:",
    int((df["osm_query_status"] == "ok").sum())
)
print(
    "Rows with OSM queried:",
    int(df["osm_queried"].sum())
)

OSM features created.
Rows with successful/usable OSM query: 46
Rows with OSM queried: 46


## 7. Add useful thermal/temporal features

These are derived from the FIRMS fields already present.

No new external data is downloaded here.

In [8]:
df["bright_ti4"] = pd.to_numeric(
    df["bright_ti4"], errors="coerce"
)
df["bright_ti5"] = pd.to_numeric(
    df["bright_ti5"], errors="coerce"
)
df["scan"] = pd.to_numeric(df["scan"], errors="coerce")
df["track"] = pd.to_numeric(df["track"], errors="coerce")

df["brightness_difference"] = (
    df["bright_ti4"] - df["bright_ti5"]
)

df["hour"] = df["acq_datetime"].dt.hour
df["day_of_week"] = df["acq_datetime"].dt.dayofweek

df["is_night"] = (
    df["daynight"].astype(str).str.upper() == "N"
).astype(int)

print("Thermal/temporal features created.")

Thermal/temporal features created.


## 8. Mainland-India quality filter

Notebook 03 introduced `in_mainland_india` because the original FIRMS region was intentionally broad enough to include neighbouring areas.

For the **ML handoff**, we exclude rows explicitly marked outside the mainland-India mask.

This does not modify the original FIRMS dataset.

In [9]:
if "in_mainland_india" in df.columns:
    before = len(df)
    df = df[df["in_mainland_india"].fillna(False)].copy()
    print(
        f"Mainland-India ML rows: {len(df):,} / {before:,}"
    )
else:
    print(
        "No in_mainland_india column found; "
        "continuing without geographic filtering."
    )

Mainland-India ML rows: 861 / 1,112


## 9. Final feature table

The **core features** below are directly compatible with the existing backend training script.

Additional engineered features are retained in the broader dataset for the ML teammate to evaluate.

In [10]:
CORE_FEATURES = [
    "frp",
    "confidence",
    "historical_count",
    "historical_frp_mean",
    "historical_frp_max",
    "persistence_days",
    "industrial_distance",
    "residential_distance",
    "road_distance",
]

ADDITIONAL_FEATURES = [
    "frp_zscore",
    "persistence_ratio",
    "has_baseline",
    "bright_ti4",
    "bright_ti5",
    "brightness_difference",
    "scan",
    "track",
    "is_night",
    "hour",
    "day_of_week",
    "near_industrial",
    "near_agriculture",
    "near_forest",
    "near_residential",
    "near_commercial",
    "osm_queried",
]

METADATA_COLUMNS = [
    "source_row_id",
    "latitude",
    "longitude",
    "lat_grid",
    "lon_grid",
    "acq_date",
    "acq_time",
    "acq_datetime",
    "confidence_raw",
    "satellite",
    "instrument",
    "daynight",
    "osm_query_status",
    "in_mainland_india",
]

ML_FEATURES = CORE_FEATURES + ADDITIONAL_FEATURES

missing_features = [
    c for c in ML_FEATURES if c not in df.columns
]

if missing_features:
    raise ValueError(
        f"Final ML feature construction is missing: {missing_features}"
    )

ml_ready = df[METADATA_COLUMNS + ML_FEATURES].copy()

print(f"ML-ready rows: {len(ml_ready):,}")
print(f"Core features: {len(CORE_FEATURES)}")
print(f"Additional features: {len(ADDITIONAL_FEATURES)}")

ML-ready rows: 861
Core features: 9
Additional features: 17


## 10. Numeric sanity checks

Before handing the data to the ML teammate, check:

- no duplicate observation IDs
- no impossible missing values in the essential core thermal fields
- numeric feature types are actually numeric
- OSM distances are non-negative where present

In [11]:
assert ml_ready["source_row_id"].is_unique

essential = [
    "frp",
    "confidence",
    "historical_count",
    "historical_frp_mean",
    "historical_frp_max",
    "persistence_days",
]

essential_missing = ml_ready[essential].isna().sum()
print("Essential feature missing values:")
display(essential_missing.to_frame("missing_count"))

distance_cols = [
    "industrial_distance",
    "residential_distance",
    "road_distance",
]

for col in distance_cols:
    invalid = (
        ml_ready[col].notna() &
        (pd.to_numeric(ml_ready[col], errors="coerce") < 0)
    ).sum()
    assert invalid == 0, f"Negative distance found in {col}"

numeric_check = {}
for col in ML_FEATURES:
    numeric_check[col] = pd.api.types.is_numeric_dtype(
        ml_ready[col]
    )

display(
    pd.DataFrame.from_dict(
        numeric_check,
        orient="index",
        columns=["numeric"]
    )
)

print("Sanity checks passed.")

Essential feature missing values:


,missing_count
frp,0
confidence,0
historical_count,0
historical_frp_mean,0
historical_frp_max,0
persistence_days,0


,numeric
frp,True
confidence,True
historical_count,True
historical_frp_mean,True
historical_frp_max,True
persistence_days,True
industrial_distance,True
residential_distance,True
road_distance,True
frp_zscore,True


Sanity checks passed.


## 11. Create the labeling template

**Do not auto-label this table.**

The existing GeoFlare training workflow requires team-assigned labels. The ML teammate should manually review the examples and fill:

- `label = 0` → `PERSISTENT_SOURCE`
- `label = 1` → `OTHER_ANOMALY`
- `label = 2` → `NEW_ABNORMAL_EVENT`

The template contains the exact core columns expected by `backend/ml/train.py`, plus useful metadata for reviewing each example.

In [12]:
label_template = ml_ready[
    [
        "source_row_id",
        "latitude",
        "longitude",
        "acq_datetime",
        "frp",
        "confidence",
        "historical_count",
        "historical_frp_mean",
        "historical_frp_max",
        "persistence_days",
        "industrial_distance",
        "residential_distance",
        "road_distance",
        "frp_zscore",
        "persistence_ratio",
        "near_industrial",
        "near_agriculture",
        "near_forest",
        "near_residential",
        "near_commercial",
        "osm_queried",
    ]
].copy()

label_template["label"] = pd.Series(
    pd.array([pd.NA] * len(label_template), dtype="Int64")
)

print(f"Labeling rows: {len(label_template):,}")
display(label_template.head(10))

Labeling rows: 861


,source_row_id,latitude,longitude,acq_datetime,frp,confidence,historical_count,historical_frp_mean,historical_frp_max,persistence_days,...,road_distance,frp_zscore,persistence_ratio,near_industrial,near_agriculture,near_forest,near_residential,near_commercial,osm_queried,label
136,136,7.99790,80.38068,2026-09-05 08:13:00,6.15,0,1.0,6.58,6.58,1.0,...,NaN,NaN,1.0,0,0,0,0,0,0,<NA>
137,137,7.99843,80.38423,2026-09-05 08:13:00,6.15,0,1.0,6.58,6.58,1.0,...,NaN,NaN,1.0,0,0,0,0,0,0,<NA>
138,138,8.01719,81.42326,2026-09-05 08:13:00,5.04,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>
139,139,8.02176,81.43032,2026-09-05 08:13:00,2.79,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>
140,140,8.02232,81.43410,2026-09-05 08:13:00,2.79,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>
141,141,8.02516,81.16928,2026-09-05 08:13:00,7.90,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>
142,142,8.02535,80.63208,2026-09-05 08:13:00,4.24,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>
143,143,8.02858,81.16877,2026-09-05 08:13:00,7.90,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>
144,144,8.02988,81.41364,2026-09-05 08:13:00,3.89,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>
145,145,8.03950,80.61161,2026-09-05 08:13:00,8.79,0,0.0,0.00,0.00,0.0,...,NaN,NaN,0.0,0,0,0,0,0,0,<NA>


## 12. Feature dictionary

This file makes the handoff understandable without requiring the ML teammate to read the entire notebook.

In [13]:
feature_dictionary = pd.DataFrame([
    ["frp", "Core", "FIRMS fire radiative power", "numeric"],
    ["confidence", "Core", "FIRMS confidence encoded n=0, l=1, h=2", "numeric"],
    ["historical_count", "Core", "Historical FIRMS detections in the same grid cell", "numeric"],
    ["historical_frp_mean", "Core", "Historical mean FRP in the same grid cell", "numeric"],
    ["historical_frp_max", "Core", "Historical maximum FRP in the same grid cell", "numeric"],
    ["persistence_days", "Core", "Number of historical active days", "numeric"],
    ["industrial_distance", "Core", "Nearest queried industrial OSM feature distance in metres", "numeric / NaN"],
    ["residential_distance", "Core", "Nearest queried residential OSM feature distance in metres", "numeric / NaN"],
    ["road_distance", "Core", "Road distance; not collected in Notebook 03", "numeric / NaN"],
    ["frp_zscore", "Additional", "Recent FRP anomaly relative to historical FRP standard deviation", "numeric / NaN"],
    ["persistence_ratio", "Additional", "Historical active-days divided by observed span", "numeric"],
    ["has_baseline", "Additional", "Whether the grid cell had historical baseline data", "0/1"],
    ["bright_ti4", "Additional", "VIIRS brightness temperature channel 4", "numeric"],
    ["bright_ti5", "Additional", "VIIRS brightness temperature channel 5", "numeric"],
    ["brightness_difference", "Additional", "bright_ti4 minus bright_ti5", "numeric"],
    ["scan", "Additional", "FIRMS scan value", "numeric"],
    ["track", "Additional", "FIRMS track value", "numeric"],
    ["is_night", "Additional", "Day/night indicator derived from FIRMS daynight", "0/1"],
    ["hour", "Additional", "Acquisition hour", "numeric"],
    ["day_of_week", "Additional", "Acquisition weekday, Monday=0", "numeric"],
    ["near_industrial", "Additional", "OSM industrial context within query radius", "0/1"],
    ["near_agriculture", "Additional", "OSM agricultural context within query radius", "0/1"],
    ["near_forest", "Additional", "OSM forest/wood context within query radius", "0/1"],
    ["near_residential", "Additional", "OSM residential context within query radius", "0/1"],
    ["near_commercial", "Additional", "OSM commercial context within query radius", "0/1"],
    ["osm_queried", "Additional", "Whether this cell was included in the OSM query set", "0/1"],
    ["label", "Target", "Manual class assigned by team", "0/1/2"],
], columns=["feature", "group", "meaning", "format"])

display(feature_dictionary)

,feature,group,meaning,format
0,frp,Core,FIRMS fire radiative power,numeric
1,confidence,Core,"FIRMS confidence encoded n=0, l=1, h=2",numeric
2,historical_count,Core,Historical FIRMS detections in the same grid cell,numeric
3,historical_frp_mean,Core,Historical mean FRP in the same grid cell,numeric
4,historical_frp_max,Core,Historical maximum FRP in the same grid cell,numeric
5,persistence_days,Core,Number of historical active days,numeric
6,industrial_distance,Core,Nearest queried industrial OSM feature distanc...,numeric / NaN
7,residential_distance,Core,Nearest queried residential OSM feature distan...,numeric / NaN
8,road_distance,Core,Road distance; not collected in Notebook 03,numeric / NaN
9,frp_zscore,Additional,Recent FRP anomaly relative to historical FRP ...,numeric / NaN


## 13. Export ML handoff files

The most important file for the ML teammate is:

`labeled_hotspots_template.csv`

It has the exact core feature names expected by the current backend training script, but its `label` column is intentionally blank.

Also exported:

- `thermotech_ml_ready_unlabeled.csv` — broad feature table + metadata
- `thermotech_ml_ready_numeric.csv` — numeric ML table + blank label
- `thermotech_feature_dictionary.csv` — definitions

In [14]:
UNLABELED_OUT = PROCESSED_DIR / "thermotech_ml_ready_unlabeled.csv"
NUMERIC_OUT = PROCESSED_DIR / "thermotech_ml_ready_numeric.csv"
LABEL_OUT = PROCESSED_DIR / "labeled_hotspots_template.csv"
DICT_OUT = PROCESSED_DIR / "thermotech_feature_dictionary.csv"

ml_ready.to_csv(UNLABELED_OUT, index=False)

numeric_out = ml_ready[ML_FEATURES].copy()
numeric_out["label"] = pd.Series(
    pd.array([pd.NA] * len(numeric_out), dtype="Int64")
)
numeric_out.to_csv(NUMERIC_OUT, index=False)

label_template.to_csv(LABEL_OUT, index=False)

feature_dictionary.to_csv(DICT_OUT, index=False)

print(f"Saved: {UNLABELED_OUT} ({len(ml_ready):,} rows)")
print(f"Saved: {NUMERIC_OUT} ({len(numeric_out):,} rows)")
print(f"Saved: {LABEL_OUT} ({len(label_template):,} rows)")
print(f"Saved: {DICT_OUT} ({len(feature_dictionary):,} rows)")

Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\thermotech_ml_ready_unlabeled.csv (861 rows)
Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\thermotech_ml_ready_numeric.csv (861 rows)
Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\labeled_hotspots_template.csv (861 rows)
Saved: c:\Users\ASUS\Documents\Sidra\ThermoTech\data\processed\thermotech_feature_dictionary.csv (27 rows)


# 14. ML handoff checkpoint ✅

### Data preparation complete

The ML teammate should receive:

```text
data/processed/
├── labeled_hotspots_template.csv
├── geoflare_ml_ready_unlabeled.csv
├── geoflare_ml_ready_numeric.csv
└── geoflare_feature_dictionary.csv
```

### Labeling

Manually assign:

```text
0 → PERSISTENT_SOURCE
1 → OTHER_ANOMALY
2 → NEW_ABNORMAL_EVENT
```

Do **not** train on rows whose `label` is blank.

### Important limitation

`road_distance` is intentionally `NaN` because Notebook 03 did not collect road context. It must **not** be fabricated from another OSM category.

The ML teammate can either:

1. train using the other available features and update `train.py`, or
2. collect road distance separately and then train with the original backend feature list.

### Recommended handoff order

1. Give the teammate `labeled_hotspots_template.csv`.
2. Have them manually review and label a sufficiently balanced set.
3. Give them `thermotech_feature_dictionary.csv`.
4. They train XGBoost on the labeled rows.
5. They evaluate the model before integrating it into the backend.
6. SHAP explanations come **after** a real model is trained.

> **No synthetic labels are created by this notebook.**